RAG Corpus Preparation (Chunking & Metadata Normalization)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Step A - Build chunks.jsonl

In [2]:
import json, re, hashlib
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ifixit_dump")
in_path = BASE / "guides.jsonl"
out_path = BASE / "chunks.jsonl"

def clean_text(s: str) -> str:
    if not s: return ""
    s = re.sub(r"<[^>]+>", " ", s)   # remove HTML
    s = re.sub(r"\s+", " ", s).strip()
    return s

def chunk_words(text, max_words=220, overlap=40):
    words = text.split()
    if not words:
        return []
    chunks = []
    step = max(1, max_words - overlap)
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i+max_words]))
        i += step
    return chunks

written = 0
with in_path.open("r", encoding="utf-8") as fin, out_path.open("w", encoding="utf-8") as fout:
    for line in fin:
        g = json.loads(line)

        guide_id = g.get("guide_id")
        title = clean_text(g.get("title") or "")
        url = g.get("url") or ""
        device = clean_text(g.get("device") or "")
        tools = g.get("tools") or []
        parts = g.get("parts") or []

        intro = clean_text(g.get("introduction") or "")
        summary = clean_text(g.get("summary") or "")
        steps = " ".join(clean_text(x) for x in (g.get("steps") or []))

        full = clean_text("\n".join([
            f"TITLE: {title}",
            f"DEVICE: {device}",
            f"SUMMARY: {summary}",
            f"INTRO: {intro}",
            f"STEPS: {steps}",
        ]))

        for idx, ch in enumerate(chunk_words(full, max_words=220, overlap=40)):
            chunk_id = hashlib.md5(f"{guide_id}-{idx}".encode()).hexdigest()
            rec = {
                "chunk_id": chunk_id,
                "guide_id": guide_id,
                "chunk_index": idx,
                "title": title,
                "url": url,
                "device": device,
                "tools": tools,
                "parts": parts,
                "text": ch,
            }
            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            written += 1

print("Wrote chunks:", written)
print("Saved to:", out_path)

Wrote chunks: 165523
Saved to: /content/drive/MyDrive/ifixit_dump/chunks.jsonl


Step B — Quick verify chunk count + sample

In [3]:
import json, random

p = "/content/drive/MyDrive/ifixit_dump/chunks.jsonl"
n = 0
samples = []
with open(p, "r", encoding="utf-8") as f:
    for line in f:
        n += 1
        if len(samples) < 3 and random.random() < 0.00002:
            samples.append(json.loads(line))
print("Total chunks:", n)
for s in samples:
    print("\n---")
    print(s["title"])
    print(s["text"][:400])

Total chunks: 165523

---
Dell Adamo Teardown
the computer itself, and instead chose to include a bulky adapter. Adamo, meet Air. Now that you're acquainted... The Adamo's dimensions, as compared to the MacBook Air: Width: 0.23&quot; larger Depth: 0.56&quot; larger Height: 0.11&quot; thinner It's interesting how the Air plays tricks with height. It certainly looks thinner than the Adamo... Both the Adamo and Air have very high torsional rigid

---
LG Neon II Speaker Replacement
a safety pin or push pin), remove the plastic screw covers along the corners of the phone. Unscrew the four 3.5 mm screws that are underneath the covers. Flip the phone over. Using the plastic opening tool, remove the front casing from the phone. You should now be able to see the interior components of the screen. When you pop the front panel off, the Call buttons will fall out. Use the plastic op

---
eMac Motherboard Assembly Replacement
TITLE: eMac Motherboard Assembly Replacement DEVICE: eMac SUMMARY: The eM

Converted 59,405 iFixit repair guides into 165,523 retrieval-ready chunks with normalized metadata for downstream vector indexing.